# The first phase for achiveing the fabric swatch using SDXL IP Adapter model and Resource Sharing (But it's failed)

In [ ]:
!pip install -q diffusers transformers accelerate peft opencv-python pillow

In [5]:
!nvidia-smi

Mon Jan 12 17:26:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
import os
import torch
import cv2
import numpy as np
from PIL import Image
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel, UNet2DConditionModel, EulerDiscreteScheduler
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

# FIX: Prevent library conflicts and optimize VRAM
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

# Configuration IDs
base_id = "stabilityai/stable-diffusion-xl-base-1.0"
repo_id = "ByteDance/SDXL-Lightning"

# 1. Load SDXL-Lightning UNet (The Fast Engine)
# We load the config from base and weights from Lightning
unet = UNet2DConditionModel.from_config(base_id, subfolder="unet").to(torch.float16)
unet.load_state_dict(load_file(hf_hub_download(repo_id, "sdxl_lightning_4step_unet.safetensors")))

# 2. Initialize the Global Pipeline (Without ControlNet initially)
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    base_id, unet=unet, controlnet=None, torch_dtype=torch.float16, variant="fp16"
)

# 3. Configure Fast Scheduler
pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config, timestep_spacing="trailing")

# 4. T4 GPU Optimizations (Always Active)
pipe.enable_model_cpu_offload() 
pipe.enable_vae_tiling() 
print("✅ Core Engine Ready (SDXL-Lightning Optimized)")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/diffusers/configuration_utils.py:250: FutureWarning: It is deprecated to pass a pretrained model name or path to `from_config`.If you were trying to load a model, please use <class 'diffusers.models.unets.unet_2d_condition.UNet2DConditionModel'>.load_config(...) followed by <class 'diffusers.models.unets.unet_2d_condition.UNet2DConditionModel'>.from_config(...) instead. Otherwise, please make sure to pass a configuration dictionary instead. This functionality will be removed in v1.0.0.
  deprecate("config-passed-as-path", "1.0.0", deprecation_message, standard_warn=False)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWar

sdxl_lightning_4step_unet.safetensors:   0%|          | 0.00/5.14G [00:00<?, ?B/s]

model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

merges.txt: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/565 [00:00<?, ?B/s]

... (12 outputs left out)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


✅ Core Engine Ready (SDXL-Lightning Optimized)


/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2213: FutureWarning: `enable_vae_tiling` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_tiling()` on a `StableDiffusionXLControlNetPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_tiling()`.
  deprecate(


In [5]:
# Load ControlNet weights individually
controlnet_canny = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16
)

def run_structural_mode(prompt, sketch_image):
    pipe.controlnet = controlnet_canny
    pipe.unload_ip_adapter() # Ensure Fabric mode is OFF
    
    canny_image = get_canny_image(sketch_image) # Using the helper function from earlier
    
    return pipe(
        prompt=prompt,
        image=canny_image,
        num_inference_steps=4,
        guidance_scale=1.0
    ).images[0]

: 

: 

: 

In [ ]:
def run_texture_mode(prompt, sketch_image, fabric_image, scale=0.7):
    # Link the Structural Control and the Texture Adapter
    pipe.controlnet = controlnet_canny
    pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl.bin")
    pipe.set_ip_adapter_scale(scale)
    
    canny_image = get_canny_image(sketch_image)
    
    image = pipe(
        prompt=prompt,
        image=canny_image,
        ip_adapter_image=fabric_image,
        num_inference_steps=4,
        guidance_scale=1.0
    ).images[0]
    
    return image

NameError: name 'sketch_img' is not defined

In [ ]:
import matplotlib.pyplot as plt

scales = [0.0, 0.5, 1.0]
generated_images = []

for s in scales:
    print(f"Testing Scale: {s}...")
    pipe.set_ip_adapter_scale(s)
    
    # Generate
    with torch.inference_mode():
        output = pipe(
            prompt="A high-fashion luxury gown, professional photography, 8k",
            image=canny_sketch,
            ip_adapter_image=fabric_swatch,
            num_inference_steps=4,
            guidance_scale=1.0
        ).images[0]
    generated_images.append(output)

# Display Results Side-by-Side
fig, axes = plt.subplots(1, 3, figsize=(20, 10))
titles = ["Scale 0.0 (No Fabric)", "Scale 0.5 (Balanced)", "Scale 1.0 (Strict Fabric)"]

for i, img in enumerate(generated_images):
    axes[i].imshow(img)
    axes[i].set_title(titles[i])
    axes[i].axis("off")

plt.tight_layout()
plt.show()

Testing Scale: 0.0


NameError: name 'canny_sketch' is not defined

# Step by Step ip adapter

In [3]:
import os
import torch
import cv2
import numpy as np
from PIL import Image
from diffusers import StableDiffusionXLControlNetPipeline, ControlNetModel, UNet2DConditionModel, EulerDiscreteScheduler
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

# FIX: Prevent library conflicts and optimize VRAM for T4
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

# Helper Function for Edges (Used in later steps)
def get_canny_image(image):
    img_np = np.array(image.resize((1024, 1024)))
    edges = cv2.Canny(img_np, 100, 200)
    edges = edges[:, :, None]
    edges = np.concatenate([edges, edges, edges], axis=2)
    return Image.fromarray(edges)

print("✅ Step 1: Environment Ready")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


✅ Step 1: Environment Ready


In [4]:
import torch
from diffusers import StableDiffusionXLControlNetPipeline, UNet2DConditionModel, EulerDiscreteScheduler
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

base_id = "stabilityai/stable-diffusion-xl-base-1.0"
repo_id = "ByteDance/SDXL-Lightning"

# Load the Lightning UNet
unet = UNet2DConditionModel.from_config(base_id, subfolder="unet").to(torch.float16)
unet.load_state_dict(load_file(hf_hub_download(repo_id, "sdxl_lightning_4step_unet.safetensors")))

# Initialize the pipeline - THIS DEFINES THE 'pipe' VARIABLE
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    base_id, unet=unet, controlnet=None, torch_dtype=torch.float16, variant="fp16"
)

pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config, timestep_spacing="trailing")
pipe.enable_model_cpu_offload() 

print("✅ 'pipe' variable is now created!")

/usr/local/lib/python3.12/dist-packages/diffusers/configuration_utils.py:250: FutureWarning: It is deprecated to pass a pretrained model name or path to `from_config`.If you were trying to load a model, please use <class 'diffusers.models.unets.unet_2d_condition.UNet2DConditionModel'>.load_config(...) followed by <class 'diffusers.models.unets.unet_2d_condition.UNet2DConditionModel'>.from_config(...) instead. Otherwise, please make sure to pass a configuration dictionary instead. This functionality will be removed in v1.0.0.
  deprecate("config-passed-as-path", "1.0.0", deprecation_message, standard_warn=False)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an is

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


✅ 'pipe' variable is now created!


In [5]:
from diffusers import ControlNetModel

# 1. Attach Structural Control
pipe.controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16
)

# 2. Attach Texture Injection
pipe.load_ip_adapter(
    "h94/IP-Adapter", 
    subfolder="sdxl_models", 
    weight_name="ip-adapter_sdxl.bin"
)
pipe.set_ip_adapter_scale(0.7)

print("✅ pipe is now upgraded with ControlNet and IP-Adapter!")

✅ pipe is now upgraded with ControlNet and IP-Adapter!


In [1]:
import torch

# 1. Clear any leftover memory from the crash
torch.cuda.empty_cache()

# 2. Re-configure the pipe for maximum memory saving
if 'pipe' in globals():
    # This is the magic command for limited VRAM (T4 GPUs)
    pipe.enable_sequential_cpu_offload() 
    pipe.enable_vae_tiling()
    print("✅ Memory protection enabled. Ready to generate.")
else:
    print("❌ Pipe not found. Please run your Model Setup cell first.")

❌ Pipe not found. Please run your Model Setup cell first.


In [3]:
import cv2
import numpy as np
from PIL import Image

def get_canny_image(image):
    image = np.array(image.resize((1024, 1024)))
    image = cv2.Canny(image, 100, 200)
    image = image[:, :, None]
    image = np.concatenate([image, image, image], axis=2)
    return Image.fromarray(image)

# Verify your filenames in sample_data
sketch_path = "/content/sample_data/fabricanova-sketch (2).png"
fabric_path = "/content/sample_data/91BxSJ-LC-L.jpg"

print("✅ Step 5: Processor and Paths Defined")

✅ Step 5: Processor and Paths Defined


In [4]:
try:
    sketch_raw = Image.open(sketch_path).convert("RGB")
    fabric_swatch = Image.open(fabric_path).convert("RGB")
    canny_sketch = get_canny_image(sketch_raw)

    print("🎨 Generating your AI design...")
    with torch.inference_mode():
        image = pipe(
            prompt="A high-fashion luxury dress, professional photography, 8k, highly detailed",
            negative_prompt="blurry, distorted, low quality",
            image=canny_sketch,
            ip_adapter_image=fabric_swatch,
            num_inference_steps=4,
            guidance_scale=1.0
        ).images[0]

    display(image)
    image.save("/content/sample_data/fabricanova_final.png")
    print("✨ Success! Result saved to sample_data.")
except Exception as e:
    print(f"❌ Error: {e}")

🎨 Generating your AI design...
❌ Error: name 'pipe' is not defined


In [37]:
!find "$PWD"

/content
/content/.config
/content/.config/configurations
/content/.config/configurations/config_default
/content/.config/.last_survey_prompt.yaml
/content/.config/default_configs.db
/content/.config/.last_opt_in_prompt.yaml
/content/.config/logs
/content/.config/logs/2025.12.09
/content/.config/logs/2025.12.09/14.41.27.893750.log
/content/.config/logs/2025.12.09/14.41.42.675750.log
/content/.config/logs/2025.12.09/14.41.33.792924.log
/content/.config/logs/2025.12.09/14.40.47.605300.log
/content/.config/logs/2025.12.09/14.41.18.717681.log
/content/.config/logs/2025.12.09/14.41.43.412452.log
/content/.config/config_sentinel
/content/.config/.last_update_check.json
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
/content/.config/active_config
/content/.config/gce
/content/sample_data
/content/sample_data/anscombe.json
/content/sample_data/README.md
/content/sample_data/fabricanova-sketch (2).png
/content/sample_data/Gemini_Generated_Image_4rc6nc4rc6nc4rc6.

# the more updated model

In [2]:
import torch
from diffusers import (
    StableDiffusionXLControlNetPipeline, 
    ControlNetModel, 
    UNet2DConditionModel, 
    EulerDiscreteScheduler
)
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

# --- CONFIGURATION ---
base_id = "stabilityai/stable-diffusion-xl-base-1.0"
repo_id = "ByteDance/SDXL-Lightning"

print("🚀 Step 1: Loading ControlNet...")
controlnet_canny = ControlNetModel.from_pretrained(
    "diffusers/controlnet-canny-sdxl-1.0", 
    torch_dtype=torch.float16
)

print("🚀 Step 2: Loading Lightning UNet...")
unet = UNet2DConditionModel.from_config(base_id, subfolder="unet").to(torch.float16)
unet.load_state_dict(load_file(hf_hub_download(repo_id, "sdxl_lightning_4step_unet.safetensors")))

print("🚀 Step 3: Building Pipeline...")
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    base_id, 
    unet=unet, 
    controlnet=controlnet_canny, 
    torch_dtype=torch.float16, 
    variant="fp16"
)

# --- MEMORY PROTECTION (CRITICAL TO PREVENT CRASH) ---
pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config, timestep_spacing="trailing")

# Instead of .to("cuda"), we use this to save memory:
pipe.enable_sequential_cpu_offload() 
pipe.enable_vae_tiling() 

print("🚀 Step 4: Loading IP-Adapter...")
pipe.load_ip_adapter(
    "h94/IP-Adapter", 
    subfolder="sdxl_models", 
    weight_name="ip-adapter_sdxl.bin"
)
pipe.set_ip_adapter_scale(0.7)

print("✅ ALL SYSTEMS READY: Pipe is defined and memory-safe.")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


🚀 Step 1: Loading ControlNet...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


🚀 Step 2: Loading Lightning UNet...


/usr/local/lib/python3.12/dist-packages/diffusers/configuration_utils.py:250: FutureWarning: It is deprecated to pass a pretrained model name or path to `from_config`.If you were trying to load a model, please use <class 'diffusers.models.unets.unet_2d_condition.UNet2DConditionModel'>.load_config(...) followed by <class 'diffusers.models.unets.unet_2d_condition.UNet2DConditionModel'>.from_config(...) instead. Otherwise, please make sure to pass a configuration dictionary instead. This functionality will be removed in v1.0.0.
  deprecate("config-passed-as-path", "1.0.0", deprecation_message, standard_warn=False)


: 

: 

: 